# 03 — Récursivité

Cas de base, pile d'appels, récurrences, exponentiation rapide, mémoïsation.

Chaque notion suit le schéma : **définition → quand l'utiliser → code testé (`assert`) → pièges → astuce**.


## 3.1 Principe, cas de base et pile d'appels

**Définition.** Une fonction récursive s'appelle elle-même sur un sous-problème plus petit, jusqu'à un cas de base terminal.

**Quand l'utiliser.** Pour « diviser pour régner » et les structures arborescentes. Éviter deux appels récursifs sur `n-1` (coût exponentiel).


In [ ]:
def factorielle(n):
    if n == 0:                    # cas de base
        return 1
    return n * factorielle(n - 1)

assert factorielle(0) == 1
assert factorielle(5) == 120

import sys
print("limite de récursion :", sys.getrecursionlimit())

def sans_cas_de_base(n):          # pas de cas de base !
    return sans_cas_de_base(n + 1)

try:
    sans_cas_de_base(0)
except RecursionError:
    print("RecursionError attrapée")
else:
    raise AssertionError("RecursionError attendue")


**Pièges.** Oublier le cas de base ; chaque appel en attente occupe un cadre de pile (espace O(profondeur)).

**Astuce.** La profondeur maximale par défaut lève `RecursionError: maximum recursion depth exceeded`.


## 3.2 Récurrences à connaître

| Relation | Complexité | Exemple |
|---|---|---|
| T(n) = T(n−1) + O(1) | O(n) | factorielle |
| T(n) = T(n−1) + O(n) | O(n²) | tri sélection récursif |
| T(n) = T(n/2) + O(1) | O(log n) | dichotomie, exponentiation rapide |
| T(n) = 2T(n/2) + O(1) | O(n) | maximum par dichotomie |
| T(n) = 2T(n/2) + O(n) | O(n log n) | tri fusion |
| T(n) = 2T(n−1) + O(1) | O(2ⁿ) | tours de Hanoï |
| T(n) = T(n−1) + T(n−2) + O(1) | O(φⁿ), φ ≈ 1,618 | Fibonacci naïf |


## 3.3 Exponentiation rapide

**Définition.** Calculer xⁿ en O(log n) multiplications en élevant au carré et en exploitant la parité de n.

**Quand l'utiliser.** Puissances d'entiers ou de matrices avec un grand exposant. Éviter la boucle naïve O(n).


In [ ]:
def puissance_rec(x, n):
    # O(log n) temps, O(log n) espace (pile d'appels)
    if n == 0:
        return 1
    y = puissance_rec(x, n // 2)     # UN SEUL appel, mémorisé
    return y * y if n % 2 == 0 else y * y * x

def puissance_iter(x, n):
    # O(log n) temps, O(1) espace
    r = 1
    while n > 0:
        if n % 2 == 1:
            r *= x
        x *= x
        n //= 2
    return r

for x in (2, 3, 5):
    for n in range(0, 20):
        assert puissance_rec(x, n) == x**n == puissance_iter(x, n)

assert puissance_rec(2, 100) == 2**100


**Pièges.** Écrire `puissance(x, n//2) * puissance(x, n//2)` : deux appels → retour en O(n).

**Astuce.** La version itérative est en espace constant O(1).


## 3.4 PGCD, Fibonacci et tours de Hanoï

**Définition.** Trois classiques de concours : algorithme d'Euclide, suite de Fibonacci, déplacement de disques.

**Quand l'utiliser.** PGCD pour simplifier une fraction ; mémoïsation dès que les sous-problèmes se répètent ; Hanoï pour illustrer le coût exponentiel.


In [ ]:
def pgcd(a, b):
    return a if b == 0 else pgcd(b, a % b)

assert pgcd(48, 18) == 6
assert pgcd(17, 5) == 1

def fib_naif(n):                  # O(φⁿ) : exponentiel !
    return n if n < 2 else fib_naif(n - 1) + fib_naif(n - 2)

def fib_memo(n, memo=None):       # O(n) grâce à la mémoïsation
    if memo is None:
        memo = {}
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

assert fib_naif(15) == fib_memo(15) == 610
assert fib_memo(90) == 2880067194370816120

def hanoi(n, depart, intermediaire, arrivee, coups):
    if n == 0:
        return
    hanoi(n - 1, depart, arrivee, intermediaire, coups)
    coups.append((depart, arrivee))
    hanoi(n - 1, intermediaire, depart, arrivee, coups)

coups = []
hanoi(4, "A", "B", "C", coups)
assert len(coups) == 2**4 - 1
assert coups[0] == ("A", "B")


**Pièges.** Fibonacci naïf recalcule les mêmes valeurs ; Hanoï double le nombre d'appels à chaque disque (2ⁿ − 1 coups).

**Astuce.** Mémoïser un dictionnaire `memo` transforme Fibonacci de O(φⁿ) en O(n).


## 3.5 Récursif ou itératif ?

**Définition.** Tout algorithme récursif peut s'écrire avec une pile explicite ; l'inverse n'est pas toujours vrai.

**Quand l'utiliser.** Récursif pour la clarté (arbres, diviser pour régner) ; itératif si la profondeur peut être grande.


In [ ]:
# factorielle itérative : O(n) temps, O(1) espace
def factorielle_iter(n):
    r = 1
    for k in range(2, n + 1):
        r *= k
    return r

for n in range(0, 15):
    assert factorielle_iter(n) == factorielle(n)

# inversion d'une liste : slicing vs boucle
def renverse_rec(L):
    return [] if not L else [L[-1]] + renverse_rec(L[:-1])

def renverse_iter(L):
    res = []
    for x in L:
        res.insert(0, x)   # O(n) par insertion → O(n²) au total
    return res

assert renverse_rec([1, 2, 3]) == [3, 2, 1]
assert renverse_iter([1, 2, 3]) == [3, 2, 1]


**Pièges.** La version récursive de `renverse` copie la liste à chaque appel (`L[:-1]`) : O(n²).

**Astuce.** En Python, `L[::-1]` inverse en O(n) sans récursion.


## Récapitulatif

| Algorithme | Complexité | Remarque |
|---|---|---|
| Factorielle | O(n) | récursif ou itératif |
| Exponentiation rapide | O(log n) | mémoïser `y` |
| PGCD (Euclide) | O(log min(a,b)) | récursif terminal |
| Fibonacci naïf | O(φⁿ) | à éviter |
| Fibonacci mémoïsé | O(n) | dictionnaire |
| Hanoï | O(2ⁿ) | 2ⁿ − 1 coups |


## Version du module

Le même algorithme, importé depuis `src/revision/`, doit donner le même résultat. Lance `pip install -e .` avant d'exécuter cette cellule.

In [ ]:
# Version du module — même résultat que le code inline ci-dessus
from revision import (
    factorielle,
    fib_memo,
    hanoi_coups,
    pgcd,
    puissance_iter,
    puissance_rec,
)

assert factorielle(5) == 120
assert puissance_rec(2, 10) == puissance_iter(2, 10) == 1024
assert pgcd(48, 18) == 6
assert fib_memo(90) == 2880067194370816120
assert len(hanoi_coups(4)) == 2**4 - 1
print("revision.recursivite + arithmetique : OK")
